# K230 TopFaceNet stage A: original CAD + Drive smoke

Associate the existing `DOASTRA_GDRIVE_OAUTH_JSON` Secret under Add-ons -> Secrets, enable Internet, then Save Version -> Save & Run All once. This stage performs storage admission before installing BlenderProc, renders exactly one original-CAD RGB/top-mask sample, audits projection geometry, uploads the complete result from Kaggle to the K230 Drive folder, and reads it back with SHA-256. It does not train or start the 200-frame preview.

Camera: small fixed-focus GC2093 lens; nominal horizontal FOV 65 degrees. Operating range: 100-400mm vertically above the top plane, 80-100 degrees to the plane. Intrinsics remain uncalibrated.


In [ ]:
"""Locate one hash-bound input across Kaggle's supported mount layouts."""
import hashlib
from pathlib import Path


def locate_input(root, manifest_sha256):
    root = Path(root).resolve()
    matches = []
    candidates = list(root.rglob("INPUT_MANIFEST.json"))
    if len(candidates) > 20:
        raise RuntimeError("Too many candidate manifests; require explicit input review")
    for candidate in candidates:
        candidate = candidate.resolve()
        if not candidate.is_relative_to(root):
            raise ValueError("Input manifest resolves outside Kaggle input mount")
        if hashlib.sha256(candidate.read_bytes()).hexdigest() == manifest_sha256:
            matches.append(candidate.parent)
    if len(matches) != 1:
        names = [str(path.relative_to(root)) for path in candidates]
        raise RuntimeError(f"Expected exactly one hash-bound input, found {len(matches)}; candidates={names}")
    return matches[0]

import sys, json
input_root=locate_input('/kaggle/input', '20e55a025d920c0f5901e83a1b63e602fdf9f0534e10861866c0844314d41bf4')
print('Verified input mount:',input_root)
sys.path.insert(0,str(input_root))
import cloud_store
from urllib.parse import urlsplit
OriginalDrive=cloud_store.Drive
class DiagnosedDrive(OriginalDrive):
    def __init__(self,*args,**kwargs):
        print('DRIVE_PHASE: SECRET_LOAD_BEGIN')
        try:
            super().__init__(*args,**kwargs)
        except Exception as error:
            chain=[]; current=error
            for _ in range(5):
                if current is None: break
                chain.append({'type':type(current).__name__,'module':type(current).__module__,
                    'http_status':getattr(current,'code',None),'errno':getattr(current,'errno',None)})
                current=current.__cause__ or current.__context__
            print('DRIVE_SECRET_ERROR_CLASSIFICATION:',json.dumps(chain))
            raise
        print('DRIVE_PHASE: SECRET_LOADED_WITHOUT_LOGGING_VALUE')
    def request(self,method,url,**kwargs):
        print('DRIVE_PHASE: GOOGLE_REQUEST',method,urlsplit(url).hostname)
        return super().request(method,url,**kwargs)
cloud_store.Drive=DiagnosedDrive
from kaggle_job import main
main('1MwltNmaI24HjvcxnG1w608qui42RAPKf',input_root,'k230-topface-smoke-20261114-a04')
